# 3. Diseño de particiones y preprocesamiento

**Objetivo:** materializar el protocolo preliminar antes de entrenar. Se usa validación
cruzada anidada: 5 particiones externas para evaluar y 3 internas para ajustar y seleccionar.
Cada caso aparece exactamente una vez en evaluación externa.

La unidad de evaluación es el caso de la misma campaña, bajo independencia provisional.
La fuente no declara lotes ni réplicas. Los identificadores únicos permiten rastrear los casos,
pero no demuestran independencia experimental. Si se confirman grupos dependientes, corresponde
repetir la evaluación con particiones agrupadas.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'experimento.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Abra el notebook desde este repositorio.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.experimento import read_config, inspect_plan, load_model_data, find_completed_run, run_experiment
from src.particiones import nested_plan
from src.modelo_baseline import build_baseline, evaluate_regression
from dataclasses import asdict

config, config_hash = read_config(ROOT / 'config' / 'model_config.yml')
X, y, trace, groups, quality, data_path, report_path = load_model_data(config)
pd.set_option('display.float_format', lambda value: f'{value:.3f}')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
print('Datos:', data_path.relative_to(ROOT))
print('Objetivo:', y.name, '[N]')

Datos: data\processed\preliminary\model_table.csv
Objetivo: First_Failure_Load_N [N]


## 3.1 Configuración y justificación registradas

In [2]:
status = inspect_plan(config)
display(pd.DataFrame([{
    'estado': status['status'], 'estrategia': config['protocol']['strategy'],
    'folds externos': config['protocol']['outer_splits'],
    'folds internos': config['protocol']['inner_splits'], 'semilla': config['protocol']['seed'],
    'familias candidatas': len(status['candidates']),
}]))
display(Markdown(config['protocol']['rationale']))
print('La configuración permite entrenamiento; esta inspección no ajusta modelos.')

,estado,estrategia,folds externos,folds internos,semilla,familias candidatas
0,ready_for_reviewed_execution,kfold,5,3,42,6


Evaluación exploratoria de casos de la misma campaña. La tabla curada contiene identificadores únicos y no declara lotes o réplicas. Se asume provisionalmente independencia entre registros; esa condición no se deduce de un identificador único. KFold barajado permite evaluar cada caso una vez y conserva suficientes observaciones en las particiones internas. La selección por MAE usa solo CV interna. No estima transferencia a otra campaña ni a familias geométricas inéditas; si se confirman dependencias debe repetirse con partición agrupada.

La configuración permite entrenamiento; esta inspección no ajusta modelos.


## 3.2 Particiones y verificación del aislamiento

In [3]:
plan = nested_plan(X, config['protocol'], groups)
display(pd.DataFrame([{
    'fold': p['fold'], 'casos entrenamiento': len(p['train']),
    'casos evaluación': len(p['test']), 'particiones internas': len(p['inner']),
    'casos reservados': ', '.join(trace.iloc[p['test']]['Case_n'].astype(str)),
} for p in plan]))
all_test = np.concatenate([p['test'] for p in plan])
assert sorted(all_test.tolist()) == list(range(len(X)))
for p in plan:
    assert not set(p['train']) & set(p['test'])
    for a, b in p['inner']:
        assert not set(p['train'][a]) & set(p['train'][b])
        assert not set(p['train'][a]) & set(p['test'])
        assert not set(p['train'][b]) & set(p['test'])
print('Verificado: cada caso se evalúa una vez; no hay solapamientos entre train y evaluación.')

,fold,casos entrenamiento,casos evaluación,particiones internas,casos reservados
0,1,20,6,3,"1, 10, 11, 13, 18, 26"
1,2,21,5,3,"3, 4, 7, 15, 25"
2,3,21,5,3,"5, 6, 14, 17, 23"
3,4,21,5,3,"19, 20, 22, 24, 27"
4,5,21,5,3,"8, 9, 12, 16, 21"


Verificado: cada caso se evalúa una vez; no hay solapamientos entre train y evaluación.


## 3.3 Preprocesamiento dentro del entrenamiento

Cada pipeline ajusta imputación de predictores, eliminación de constantes y escalamiento
dentro de su propio conjunto de entrenamiento. La carga de primera falla nunca se imputa.
Los identificadores y las respuestas posensayo no entran en X.

In [4]:
from src.modelos import candidate_specs
specs = candidate_specs(config)
display(pd.DataFrame([{
    'familia': family,
    'imputación': 'mediana de train',
    'constantes': 'eliminación dentro de train',
    'escalamiento': type(pipe.named_steps['scale']).__name__ if pipe.named_steps['scale'] != 'passthrough' else 'no requerido',
    'estimador': type(pipe.named_steps['model']).__name__,
} for family, (pipe, _) in specs.items()]))
assert y.name not in X.columns and 'Case_n' not in X.columns
print('X contiene exclusivamente los predictores declarados en el esquema.')

,familia,imputación,constantes,escalamiento,estimador
0,ridge,mediana de train,eliminación dentro de train,StandardScaler,Ridge
1,elastic_net,mediana de train,eliminación dentro de train,StandardScaler,ElasticNet
2,support_vector,mediana de train,eliminación dentro de train,StandardScaler,SVR
3,decision_tree,mediana de train,eliminación dentro de train,no requerido,DecisionTreeRegressor
4,random_forest,mediana de train,eliminación dentro de train,no requerido,RandomForestRegressor
5,gaussian_process,mediana de train,eliminación dentro de train,StandardScaler,GaussianProcessRegressor


X contiene exclusivamente los predictores declarados en el esquema.


## 3.4 Alcance de esta evaluación

Este protocolo aproxima la predicción de otros casos comparables de la misma campaña.
No mide transferencia a nuevas campañas, materiales o configuraciones completamente inéditas.
El EDA ya se realizó sobre este conjunto, por lo que esta evaluación conserva carácter preliminar.

Siguiente: [04 · Baseline de referencia](04_Baseline_de_Referencia.ipynb).